In [4]:
import os
import json

## Convert data format of current open_flamingo to instruct_flamingo

In [12]:
def convert_single_json_obj(source_json):
    """
    Example of source_json:
    {
        "image": "11590.jpg",
        "id": "11590",
        "conversations": [
            {
                "from": "human",
                "value": "<image>How would you determine the pathological impact?\n"
            },
            {
                "from": "gpt",
                "value": "\n\n+ Reasoning:\n- Step 1: Recognize the disease area\n- Answer: ```json\n[{\"bbox_2d\": [37, 242, 122, 405], \"label\": \"disease area\"}, {\"bbox_2d\": [113, 244, 199, 400], \"label\": \"disease area\"}]\n```\n- Step 2: How would you specify the visible traits of this lesion?\n- Answer: Substantial sulcal widening of posterior cingulate and parieto-occipital sulci, substantial gyral atrophy.\n- Step 3: What is the stage of the lesion?\n- Answer: Koedam = 2\n\n+ Final Answer: Moderate-Dementia"
            }
        ]
    }

    Example of target_json:
    {
        "input": "An Instruction or a question. Image path(s) (either absolute or relative) can be interleaved here as <img_path>path/to/the/image.png<img_path>, there can be more than one images: <img_path>path/to/the/second/image.png<img_path>",
        "output": "Expected response or answer. The language modelling loss only operate on this part, and it contains text only."
    }
    """
    img_rel_path = source_json["image"]
    new_img_tag = f"<img_path>{img_rel_path}<img_path>"
    
    old_input = source_json["conversations"][0]["value"]

    # Remove \n in the end, move new_img_tag to the end of new_input
    new_input = old_input.replace("\n", "").replace("<image>", "")
    new_input = new_input + new_img_tag

    target_json = {
        "input": new_input,
        "output": source_json["conversations"][1]["value"]
    }

    return target_json

def convert_json_file(source_json_fp, target_dir="./inst_flam_format"):
    print(f"Converting JSON file {source_json_fp}")
    with open(source_json_fp, "r") as f:
        data = json.load(f)

    new_data = [convert_single_json_obj(entry) for entry in data]

    target_json_fn = source_json_fp.split("/")[-1].replace("refined", "refined_converted")

    target_json_fp = os.path.join(target_dir, target_json_fn)
    os.makedirs(target_dir, exist_ok=True)

    with open(target_json_fp, "w") as f:
        json.dump(new_data, f, indent=4)

    print(f"Converted JSON saved to {target_json_fp}")
    return target_json_fp

In [13]:
train_data_fp = "/app/baseline_models/sample_data/llama_mri_cot/llava_med_mri_bbox_train_CoT_new_refined.json"
test_data_fp = "/app/baseline_models/sample_data/llama_mri_cot/llava_med_mri_bbox_test_CoT_new_refined.json"
val_data_fp = "/app/baseline_models/sample_data/llama_mri_cot/llava_med_mri_bbox_val_CoT_new_refined.json"

convert_json_file(train_data_fp)
convert_json_file(test_data_fp)
convert_json_file(val_data_fp)

Converting JSON file /app/baseline_models/sample_data/llama_mri_cot/llava_med_mri_bbox_train_CoT_new_refined.json
Converted JSON saved to ./inst_flam_format/llava_med_mri_bbox_train_CoT_new_refined_converted.json
Converting JSON file /app/baseline_models/sample_data/llama_mri_cot/llava_med_mri_bbox_test_CoT_new_refined.json
Converted JSON saved to ./inst_flam_format/llava_med_mri_bbox_test_CoT_new_refined_converted.json
Converting JSON file /app/baseline_models/sample_data/llama_mri_cot/llava_med_mri_bbox_val_CoT_new_refined.json
Converted JSON saved to ./inst_flam_format/llava_med_mri_bbox_val_CoT_new_refined_converted.json


'./inst_flam_format/llava_med_mri_bbox_val_CoT_new_refined_converted.json'

## Create test dataset

In [2]:
test_data_fp = "/mnt/data/nict/maund/baseline_models/sample_data/llama_mri_cot/instruct_flamingo/converted_dataset/llava_med_mri_bbox_test_CoT_new_refined_converted.json"

In [5]:
def prepare_eval_data(test_data_fp, cot_inst=True):

    with open(test_data_fp, "r") as f:
        test_data = json.load(f)

    eval_data = []

    for entry in test_data:
        current_input = entry["input"]
        current_output = entry["output"]

        new_cot = current_output.split("Final Answer:")[0]
        new_answer = entry["output"].split("Final Answer:")[-1].replace(" ", "")
        if cot_inst:
            new_question = current_input + "\n" + new_cot + " Final Answer: "
        else:
            new_question = current_input

        eval_data.append({
            "input": new_question,
            "output": new_answer
        })

    inst_form = "long" if cot_inst else "short"

    eval_data_fp = f"/mnt/data/nict/maund/baseline_models/sample_data/llama_mri_cot/instruct_flamingo/converted_dataset/llava_med_mri_bbox_eval_{inst_form}_CoT_new_refined_converted.json"
    with open(eval_data_fp, "w") as f:
        json.dump(eval_data, f, indent=4)


prepare_eval_data(test_data_fp, cot_inst=False)

### Extract keyword-answer in long answer

In [1]:
import re

def extract_answer_regex(text: str) -> str | None:
    """
    Extracts a hyphenated word following 'Final Answer: ' using regex.

    Args:
        text: The input string to search.

    Returns:
        The extracted word (e.g., 'Mild-Dementia') or None if no match is found.
    """
    # This pattern looks for "Final Answer:", followed by optional whitespace,
    # and then captures a sequence of word characters and hyphens.
    match = re.search(r"Final Answer:\s*([\w-]+)", text)
    
    if match:
        return match.group(1) # Return the first captured group
    return None

# --- Examples ---
string1 = "The model's conclusion is Final Answer: Mild-Dementia"
string2 = "Final Answer: Moderate-Dementia."
string3 = "Final Answer:No-Hyphen"
string4 = "This string has no answer."

print(f"'{string1}' -> '{extract_answer_regex(string1)}'")
print(f"'{string2}' -> '{extract_answer_regex(string2)}'")
print(f"'{string3}' -> '{extract_answer_regex(string3)}'")
print(f"'{string4}' -> '{extract_answer_regex(string4)}'")

'The model's conclusion is Final Answer: Mild-Dementia' -> 'Mild-Dementia'
'Final Answer: Moderate-Dementia.' -> 'Moderate-Dementia'
'Final Answer:No-Hyphen' -> 'No-Hyphen'
'This string has no answer.' -> 'None'
